In [10]:
%load_ext autoreload
%autoreload 2
%env ANYWIDGET_HMR=1

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
env: ANYWIDGET_HMR=1


# searchselect

A searchable multi-select picker for any list of strings. It knows nothing but strings — column names, category values, file paths, whatever you have a list of.

In [ ]:
import random
import string

from searchselect import SearchSelect


def randomword(length: int = 10) -> str:
    letters = string.ascii_lowercase + string.ascii_uppercase
    return "".join(random.choice(letters) for _ in range(length))


test_items = [f"Item{i}" for i in range(1, 100 + 1)] + [randomword() for _ in range(100)]
random.shuffle(test_items)

picker = SearchSelect(items=test_items)
picker

## Reading the result

`selected` is an **explicit choice** — the items you ticked. Tick a few above, then re-run this cell.

Filtering never changes it: tick something, then type a search that excludes it, and it stays selected. The search box changes what you can *see*, not what you have *chosen*.

In [12]:
picker.selected

[]

`filtered` is a **query result** — whatever the search box currently matches. With an empty search box it is every item, never an empty list.

That makes it a bulk-select: type a pattern, take all the matches, no clicking.

In [13]:
len(picker.filtered), picker.filtered[:5]

(200, ['Item58', 'Item72', 'aRKjgBcatv', 'Item60', 'Item80'])

`query` is writable too, so you can drive the search box from Python. Combined with the header checkbox — which selects **everything matching the current query**, not just what's on screen — that's the click-free path to a large selection.

Note that the frontend does the matching, so `filtered` updates one comm round trip later. Set the query in one cell and read `filtered` from the *next* one; reading it in the same cell gives you the previous value.

In [ ]:
picker.query = "Item1"  # watch the search box above

In [ ]:
len(picker.filtered)  # re-run after the widget has re-filtered

In [ ]:
picker.query = ""

## Writing back

`selected` is writable, so you can preselect or clear it from Python. The checkboxes follow. Values that aren't in `items` are ignored.

In [14]:
picker.selected = ["Item1", "Item2", "not-a-real-item"]
picker.selected

['Item1', 'Item2']

In [15]:
picker.selected = []
picker.selected

[]

Selection is keyed on the item string, not on its row position. So replacing `items` keeps ticks on items that still exist and drops the rest — it never silently transfers a tick to whatever moved into that row.

In [16]:
picker.selected = ["Item1", "Item2"]
picker.items = ["Item2", "Item3"]

picker.selected  # Item1 is gone, Item2 survived

['Item2']

## Using it

The widget has no dataframe dependency — it hands back a plain `list[str]`, so it composes with whatever you use. Picking columns out of a wide frame is the obvious case.

In [17]:
import polars as pl

df = pl.DataFrame({name: [1, 2, 3] for name in test_items[:20]})

columns = SearchSelect(items=df.columns)
columns

In [18]:
df.select(columns.selected) if columns.selected else df.head()

Item58,Item72,aRKjgBcatv,Item60,Item80,rXXXclSuxF,Item12,Item90,uvkEdvQNWc,kLUwqkjkHs,Item62,Item68,gmxrlqSLCk,uYqtNYYqAU,vKZoLohzwO,Item3,YrdjVObJLj,Item47,Item8,Item34
i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1
2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2,2
3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,3
